# Train Hierarchical Frame Classifier

This notebook trains a simple hierarchical classifier over shared sentence-transformer embeddings: one substantive-discourse head over all labels, plus clinical and lived-experience heads trained only on substantive examples. The held-out human validation set is used only for evaluation.


In [ ]:
from __future__ import annotations

import json
import pickle
from pathlib import Path

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "01_classification":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

CLASSIFICATION_DIR = PROJECT_ROOT / "data/interim/lsc/classification"
HUMAN_LABELS_PATH = CLASSIFICATION_DIR / "human_labels/frame_human_labels.csv"
CORRECTED_LLM_PATH = CLASSIFICATION_DIR / "human_correction/frame_llm_correction_completed.csv"
MODEL_DIR = PROJECT_ROOT / "data/processed/lsc/classification"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_MODEL = "sentence-transformers/all-mpnet-base-v2"
CLASSIFIER_VERSION = "hierarchical_frame_classifier_v1"
CODEBOOK_VERSION = "v0.4"
PROMPT_VERSION = "annotator_v4+critic_v5"


## Load Training and Validation Labels

The 200-case validation split remains human-only. ACT-corrected LLM labels can be added to training once the correction sheet has been completed.

In [ ]:
if not HUMAN_LABELS_PATH.exists():
    print(f"No human labels found yet: {HUMAN_LABELS_PATH.relative_to(PROJECT_ROOT)}")
    raise SystemExit("Complete and ingest human annotations first.")

human = pd.read_csv(HUMAN_LABELS_PATH)
pilot = human.loc[human["annotation_round"].eq("pilot")].copy().reset_index(drop=True)
validation = human.loc[human["annotation_round"].eq("validation")].copy().reset_index(drop=True)

training_parts = [pilot]
if CORRECTED_LLM_PATH.exists():
    corrected = pd.read_csv(CORRECTED_LLM_PATH)
    corrected = corrected[
        [
            "annotation_id",
            "context_id",
            "analysis_unit",
            "lsc_year",
            "raw_form",
            "target_sentence_plus_adjacent",
            "corrected_substantive_target_discourse",
            "corrected_clinical_frame_present",
            "corrected_lived_experience_frame_present",
            "corrected_confidence",
        ]
    ].rename(
        columns={
            "corrected_substantive_target_discourse": "substantive_target_discourse",
            "corrected_clinical_frame_present": "clinical_frame_present",
            "corrected_lived_experience_frame_present": "lived_experience_frame_present",
            "corrected_confidence": "confidence",
        }
    )
    training_parts.append(corrected)
else:
    print(f"No corrected LLM labels found yet: {CORRECTED_LLM_PATH.relative_to(PROJECT_ROOT)}")
    print("Training will use the human pilot only.")

train = pd.concat(training_parts, ignore_index=True)


def parse_bool_or_na(value: object) -> bool | pd.NA:
    if pd.isna(value):
        return pd.NA
    text = str(value).strip().lower()
    if text in {"", "na", "n/a", "none", "nan"}:
        return pd.NA
    if text in {"true", "t", "yes", "y", "1"}:
        return True
    if text in {"false", "f", "no", "n", "0"}:
        return False
    return pd.NA

for frame in [train, validation]:
    for column in ["substantive_target_discourse", "clinical_frame_present", "lived_experience_frame_present"]:
        frame[column] = frame[column].map(parse_bool_or_na).astype("boolean")

if validation.empty:
    raise ValueError("Validation set is empty; do not train without held-out human validation labels.")

for name, frame in [("train", train), ("validation", validation)]:
    if frame["substantive_target_discourse"].isna().any():
        raise ValueError(f"{name} has missing Stage-0 labels.")
    substantive = frame["substantive_target_discourse"].eq(True)
    if frame.loc[substantive, ["clinical_frame_present", "lived_experience_frame_present"]].isna().any().any():
        raise ValueError(f"{name} has missing Stage-1 labels among substantive rows.")

print(f"Training rows: {len(train):,}")
print(f"Validation rows: {len(validation):,}")
print(f"Substantive training rows for frame heads: {int(train['substantive_target_discourse'].sum()):,}")


## Embed Passages

The classifier sees the target group and passage text only. Year, domain, and URL are excluded to reduce leakage.

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
except ImportError as error:
    raise ImportError("Install the msc-nlp environment with sentence-transformers before training.") from error

def classifier_text(frame: pd.DataFrame) -> list[str]:
    return [
        f"TARGET={row.analysis_unit}\nPASSAGE={row.target_sentence_plus_adjacent}"
        for row in frame.itertuples(index=False)
    ]

embedder = SentenceTransformer(EMBEDDING_MODEL)
x_train = embedder.encode(classifier_text(train), normalize_embeddings=True, show_progress_bar=True)
x_validation = embedder.encode(classifier_text(validation), normalize_embeddings=True, show_progress_bar=True)

## Train and Evaluate Hierarchical Heads

The substantive head is trained on every labelled example. The clinical and lived-experience heads are trained and evaluated only on rows where `substantive_target_discourse = TRUE`.


In [ ]:
def derive_frame_from_predictions(substantive: bool, clinical: bool | pd.NA, lived: bool | pd.NA) -> str:
    if not substantive:
        return "non_substantive_or_insufficient"
    clinical_bool = bool(clinical)
    lived_bool = bool(lived)
    if clinical_bool and lived_bool:
        return "mixed"
    if clinical_bool:
        return "clinical_only"
    if lived_bool:
        return "lived_only"
    return "substantive_other"


def train_logistic_head(x, y, head_name: str) -> Pipeline:
    if y.nunique() < 2:
        raise ValueError(f"Training labels for {head_name} contain only one class.")
    model = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=20260604)),
        ]
    )
    model.fit(x, y)
    return model

models = {}
metrics = []
validation_predictions = validation[["annotation_id", "context_id", "analysis_unit", "lsc_year", "derived_frame"]].copy()

head_specs = [
    ("substantive_target_discourse", "p_substantive", "predicted_substantive_target_discourse", train.index, validation.index),
    (
        "clinical_frame_present",
        "p_clinical_given_substantive",
        "predicted_clinical_frame_present",
        train.index[train["substantive_target_discourse"].eq(True)],
        validation.index[validation["substantive_target_discourse"].eq(True)],
    ),
    (
        "lived_experience_frame_present",
        "p_lived_given_substantive",
        "predicted_lived_experience_frame_present",
        train.index[train["substantive_target_discourse"].eq(True)],
        validation.index[validation["substantive_target_discourse"].eq(True)],
    ),
]

for label_column, probability_column, prediction_column, train_index, validation_index in head_specs:
    y_train = train.loc[train_index, label_column].astype(bool).astype(int)
    y_validation = validation.loc[validation_index, label_column].astype(bool).astype(int)
    model = train_logistic_head(x_train[train_index], y_train, label_column)
    y_pred = model.predict(x_validation[validation_index])
    y_prob = model.predict_proba(x_validation[validation_index])[:, 1]
    precision, recall, f1, support = precision_recall_fscore_support(y_validation, y_pred, average="binary", zero_division=0)
    metrics.append({"head": label_column, "precision": precision, "recall": recall, "f1": f1, "support_positive": int(y_validation.sum()), "support_total": int(len(y_validation))})
    validation_predictions[probability_column] = pd.NA
    validation_predictions[prediction_column] = pd.NA
    validation_predictions.loc[validation_index, probability_column] = y_prob
    validation_predictions.loc[validation_index, prediction_column] = y_pred.astype(bool)
    models[label_column] = model
    print(label_column)
    print(classification_report(y_validation, y_pred, zero_division=0))
    print(confusion_matrix(y_validation, y_pred))

non_substantive_validation = validation_predictions["predicted_substantive_target_discourse"].eq(False)
validation_predictions.loc[non_substantive_validation, "predicted_clinical_frame_present"] = pd.NA
validation_predictions.loc[non_substantive_validation, "predicted_lived_experience_frame_present"] = pd.NA
validation_predictions["predicted_derived_frame"] = [
    derive_frame_from_predictions(substantive, clinical, lived)
    for substantive, clinical, lived in zip(
        validation_predictions["predicted_substantive_target_discourse"],
        validation_predictions["predicted_clinical_frame_present"],
        validation_predictions["predicted_lived_experience_frame_present"],
    )
]
derived_macro_f1 = f1_score(validation_predictions["derived_frame"], validation_predictions["predicted_derived_frame"], average="macro", zero_division=0)
metrics.append({"head": "derived_frame_macro", "precision": pd.NA, "recall": pd.NA, "f1": derived_macro_f1, "support_positive": pd.NA, "support_total": len(validation_predictions)})

metrics_df = pd.DataFrame(metrics)
metrics_df


## Save Model and Evaluation Outputs

In [ ]:
model_path = MODEL_DIR / "hierarchical_frame_logistic_models.pkl"
metrics_path = MODEL_DIR / "frame_classifier_validation_metrics.csv"
predictions_path = MODEL_DIR / "frame_classifier_validation_predictions.csv"
metadata_path = MODEL_DIR / "frame_classifier_metadata.json"

with model_path.open("wb") as handle:
    pickle.dump(models, handle)

metrics_df.to_csv(metrics_path, index=False)
validation_predictions.to_csv(predictions_path, index=False)

metadata = {
    "classifier_version": CLASSIFIER_VERSION,
    "embedding_model": EMBEDDING_MODEL,
    "classifier": "three logistic heads over shared sentence-transformer embeddings; clinical/lived heads trained only on substantive rows",
    "codebook_version": CODEBOOK_VERSION,
    "prompt_version": PROMPT_VERSION,
    "training_rows": int(len(train)),
    "substantive_training_rows": int(train["substantive_target_discourse"].sum()),
    "validation_rows": int(len(validation)),
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print("Wrote classifier outputs:")
for path in [model_path, metrics_path, predictions_path, metadata_path]:
    print(f"- {path.relative_to(PROJECT_ROOT)}")
